## Example 1: Parse a connection string into a BlobServiceClient

In [ ]:
from azure.storage.blob import BlobServiceClient

# Example 1: Parse a connection string into a BlobServiceClient
# Building a client does not contact the service, so a made-up account name and key are enough.
ACCOUNT_NAME = "teststorage"
ACCOUNT_KEY = "dGVzdGtleQ=="  # base64 of b"testkey", not a real key
CONNECTION_STRING = (f"DefaultEndpointsProtocol=https;AccountName={ACCOUNT_NAME};"
                     f"AccountKey={ACCOUNT_KEY};EndpointSuffix=core.windows.net")

service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
assert service_client.account_name == ACCOUNT_NAME
assert service_client.url == f"https://{ACCOUNT_NAME}.blob.core.windows.net/"
assert service_client.primary_hostname == f"{ACCOUNT_NAME}.blob.core.windows.net"
print(f"account: {service_client.account_name}  url: {service_client.url}")

## Example 2: Container and blob clients built from names and URLs

In [ ]:
from azure.storage.blob import BlobClient, ContainerClient

# Example 2: Container and blob clients built from names and URLs
account_url = "https://teststorage.blob.core.windows.net"
container_client = ContainerClient(account_url=account_url, container_name="test-container",
                                   credential="placeholder-key")
assert container_client.container_name == "test-container"
assert container_client.account_name == "teststorage"
assert container_client.url == f"{account_url}/test-container"

blob_client = container_client.get_blob_client("reports/2024/summary.txt")
assert (blob_client.container_name, blob_client.blob_name) == ("test-container", "reports/2024/summary.txt")
assert blob_client.url == f"{account_url}/test-container/reports/2024/summary.txt"

# from_blob_url splits a URL back into account, container and blob name, decoding %20 and friends
parsed = BlobClient.from_blob_url(f"{account_url}/test-container/dir/my%20file.txt")
assert (parsed.account_name, parsed.container_name, parsed.blob_name) == ("teststorage", "test-container", "dir/my file.txt")
print("blob URL:", blob_client.url)
print("parsed blob name:", parsed.blob_name)

## Example 3: Blob types and access tiers

In [ ]:
from azure.storage.blob import BlobType, StandardBlobTier

# Example 3: Blob types and access tiers
# The enum values are the strings the service uses in its x-ms-blob-type and x-ms-access-tier headers.
assert {t.value for t in BlobType} >= {"BlockBlob", "PageBlob", "AppendBlob"}
assert BlobType("BlockBlob") is BlobType.BLOCKBLOB
tiers = {t.value for t in StandardBlobTier}
assert {"Hot", "Cool", "Archive"} <= tiers, tiers
assert StandardBlobTier("Hot") is StandardBlobTier.HOT
print("blob types:", sorted(t.value for t in BlobType))
print("access tiers:", sorted(tiers))

## Example 4: Content settings and SAS permission strings

In [ ]:
from azure.storage.blob import BlobSasPermissions, ContentSettings

# Example 4: Content settings and SAS permission strings
settings = ContentSettings(
    content_type="text/plain",
    content_encoding="utf-8",
    content_language="en-US",
    content_disposition='attachment; filename="test.txt"',
    cache_control="max-age=3600",
)
# ContentSettings is dict-like, so it round-trips through a plain dict
copy = ContentSettings(**dict(settings))
assert copy == settings and dict(copy) == dict(settings)
assert (copy.content_type, copy.cache_control) == ("text/plain", "max-age=3600")

permissions = BlobSasPermissions(read=True, write=True, create=True)
assert str(permissions) == "rcw", str(permissions)  # the service's fixed letter order
parsed = BlobSasPermissions.from_string("rw")
assert (parsed.read, parsed.write, parsed.delete) == (True, True, False)
print("content settings:", dict(settings))
print("permission string:", str(permissions))

## Example 5: Generate a SAS token for one blob

In [ ]:
from datetime import datetime, timedelta, timezone
from urllib.parse import parse_qs

from azure.storage.blob import BlobClient, BlobSasPermissions, generate_blob_sas

# Example 5: Generate a SAS token for one blob
# Signing happens locally with the account key from Example 1; nothing is sent to the service.
expiry = datetime.now(timezone.utc) + timedelta(hours=1)
token = generate_blob_sas(
    account_name=ACCOUNT_NAME,
    container_name="reports",
    blob_name="2024/summary.txt",
    account_key=ACCOUNT_KEY,
    permission=BlobSasPermissions(read=True),
    expiry=expiry,
)
fields = parse_qs(token)
assert "sig=" in token and fields["sig"][0], token
assert fields["sp"] == ["r"] and fields["sr"] == ["b"], fields  # read-only, scoped to one blob
assert fields["se"] == [expiry.strftime("%Y-%m-%dT%H:%M:%SZ")], fields["se"]

sas_url = f"https://{ACCOUNT_NAME}.blob.core.windows.net/reports/2024/summary.txt?{token}"
sas_client = BlobClient.from_blob_url(sas_url)
assert (sas_client.container_name, sas_client.blob_name) == ("reports", "2024/summary.txt")
print("SAS fields:", sorted(fields))